# Update plants csv

In [21]:
import pandas as pd
from openai import AsyncOpenAI
from dotenv import load_dotenv
import asyncio

In [22]:
df = pd.read_csv('Plants_Formatted.csv', encoding='ISO-8859-1')
df = df[['Scientific Name', 'Common Name']].rename(
    columns={
        'Scientific Name': 'scientific_name',
        'Common Name': 'common_name'
    }
)
df['description'] = ''
df['image'] = ''
df.head()

,scientific_name,common_name,description,image
0,Adiantum peruvianum,Silver-Dollar Fern,,
1,Adiantum raddianum,Maidenhair Fern,,
2,Adiantum ternerum 'Scutum Roseum',Rosy Maidenhair Fern,,
3,Adiantum trapeziforme,Diamond Maidenhair Fern,,
4,Aechmea 'Better than Bert',Living Vase Bromeliad,,


In [24]:
load_dotenv()

client = AsyncOpenAI()

async def get_desc(scientific_name):
    try:
        response = await client.chat.completions.create(
            model='gpt-5-mini',
            messages=[
                {
                    'role': 'user', 
                    'content': f'Generate 30-50 word easy to understand description for the following plant species: {scientific_name}'}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f'Error: {e}')

async def main():
    tasks = [get_desc(row['scientific_name']) for index, row in df.iterrows()]
    descriptions = await asyncio.gather(*tasks)
    df['description'] = descriptions
    df.to_csv('plants_data.csv', index=False)

await main()

print(df.head())

                     scientific_name              common_name  \
0                Adiantum peruvianum       Silver-Dollar Fern   
1                 Adiantum raddianum          Maidenhair Fern   
2  Adiantum ternerum 'Scutum Roseum'     Rosy Maidenhair Fern   
3              Adiantum trapeziforme  Diamond Maidenhair Fern   
4         Aechmea 'Better than Bert'    Living Vase Bromeliad   

                                         description image  
0  Adiantum peruvianum, the Peruvian maidenhair f...        
1  Adiantum raddianum, the maidenhair fern, has d...        
2  Adiantum tenerum 'Scutum Roseum' is a delicate...        
3  Adiantum trapeziforme is a delicate maidenhair...        
4  Aechmea 'Better than Bert' is a striking brome...        


In [26]:
df.to_csv('plants_data.csv', index=False)

In [28]:
df.drop_duplicates(subset=['scientific_name'], keep='first', inplace=True)

In [29]:
df.to_csv('plants_data_clean.csv', index=False)